# Fine-Tuning OLMo-3 7B Instruct with 4-bit QAT (mxfp4) + LoRA on Apple Silicon

This notebook walks through **Quantization-Aware Training (QAT)** combined with **LoRA** fine-tuning on [OLMo-3 7B Instruct](https://huggingface.co/allenai/Olmo-3-7B-Instruct) using [`mlx-lm-lora`](https://github.com/Goekdeniz-Guelmez/mlx-lm-lora) on Apple Silicon (M-series chips).

---

### Why QAT instead of Post-Training Quantization (PTQ)?

Standard **PTQ** snaps a trained model's weights to lower precision after training — it's fast and requires no data, but quality degrades noticeably at 4-bit, especially for reasoning and instruction-following.

**QAT** simulates quantization *during* training using a straight-through estimator (STE). The forward pass uses quantized weights, but gradients still flow through. This teaches the model to distribute its weight values in a way that survives low-precision rounding, recovering most of the lost quality at a fraction of the full-precision model's memory footprint.

### Why mxfp4?

`mxfp4` is Apple's **MX (Microscaling) FP4** format, optimized for the Neural Engine on M-series chips. Unlike INT4, which maps all values to a uniform integer grid, MX formats use a per-group shared exponent. This gives better dynamic range, reduces outlier damage, and translates directly to faster on-device inference via Metal Performance Shaders.

### Why LoRA on top of QAT?

We don't full-fine-tune all 7B weights — that would require far more memory and time than is practical. Instead, **LoRA** injects small trainable rank-decomposition matrices alongside the frozen (quantized) base weights. During QAT the LoRA deltas are trained in full precision while the base weights are kept quantized, so the adapter learns to steer the model's outputs in a way that compensates for quantization error.

### Why use the same dataset as the base model?

The training set here — `mlx-community/Dolci-Instruct-SFT-No-Tools-100K` — is the **same dataset used to fine-tune the original full-precision OLMo-3 7B Instruct model**. Replaying the original training distribution during QAT is intentional: it gives the model the best possible chance to recover its pre-quantization capabilities, because it is re-learning on exactly the data that shaped its weights in the first place, rather than adapting to a shifted distribution at the same time as adapting to precision loss.

In [ ]:
%%capture
!pip install -U mlx-lm-lora

## Step 1 — Imports

We pull in everything we need from `mlx-lm-lora` and its dependencies:

| Import | Purpose |
|---|---|
| `from_pretrained` | Loads the base model with optional quantization and creates the LoRA adapter file |
| `save_pretrained_merged` | Merges the LoRA adapter back into the base weights and saves the result |
| `calculate_iters` | Convenience helper: computes the total training iteration count from dataset size, batch size, and desired epochs |
| `push_to_hub` | Uploads the final model to the Hugging Face Hub |
| `SFTTrainingArgs` / `train_sft` / `evaluate_sft` | SFT (Supervised Fine-Tuning) training loop and argument dataclass — this is where QAT is configured |
| `CacheDataset` / `TextDataset` | Lightweight dataset wrappers that tokenize on the fly and cache sequences in RAM for fast iteration |
| `print_trainable_parameters` | Shows how many parameters are trainable vs frozen — useful to verify the LoRA rank is applied correctly |
| `TrainingCallback` / `WandBCallback` | Hooks called at each logging/eval step; swap `TrainingCallback` for `WandBCallback` to log to Weights & Biases |
| `generate` | MLX inference function used to test the model before and after training |
| `mlx.optimizers` | MLX optimizer module — we use `AdamW` for weight-decay-regularized LoRA updates |
| `load_dataset` | HuggingFace `datasets` loader for pulling data from the Hub |

In [ ]:
from mlx_lm_lora.utils import from_pretrained, save_pretrained_merged, calculate_iters, push_to_hub
from mlx_lm_lora.trainer.sft_trainer import SFTTrainingArgs, train_sft, evaluate_sft
from mlx_lm_lora.trainer.datasets import CacheDataset, TextDataset

from mlx_lm.tuner.utils import print_trainable_parameters
from mlx_lm.tuner.callbacks import TrainingCallback, WandBCallback
from mlx_lm.generate import generate

import mlx.optimizers as optim

from datasets import load_dataset

## Step 2 — Configuration

All key hyperparameters are set here so they propagate through the rest of the notebook consistently.

### LoRA configuration

| Parameter | Value | Why |
|---|---|---|
| `rank` | `12` | Controls the size of the low-rank bottleneck matrices. Rank 12 sits between the cheaper rank 8 and the more expressive rank 16 — a good default for a 7B model doing SFT. Higher ranks recover more of the model's expressive capacity but cost more memory and time. |
| `dropout` | `0.0` | LoRA dropout acts as regularisation. With a small dataset (100 samples in quick mode) overfitting is the bigger risk, but dropout can destabilise QAT. Keep at 0.0 here; increase to 0.05–0.1 only if you train on thousands of samples. |
| `scale` | `10.0` | Scales the magnitude of the LoRA update before it is added to the base weights. A higher value makes each LoRA step hit harder. 10.0 is aggressive; reduce to 4–8 if you see training instability. |
| `use_dora` | `False` | DoRA decomposes the weight update into magnitude and direction components, which can improve convergence. Disabled here to keep the training loop simple and memory use low. Enable if rank alone is not giving enough flexibility. |
| `num_layers` | `10` | Only the last 10 transformer blocks get LoRA adapters. For QAT the lower layers mostly need to absorb quantization noise structurally — the upper layers drive output quality, so targeting them is efficient. Use `-1` to apply LoRA to all layers. |

### Quantization configuration

| Parameter | Value | Why |
|---|---|---|
| `bits` | `4` | 4-bit reduces model memory by ~4× vs float16 with minimal quality loss when QAT is used. 8-bit is safer but larger; 4-bit is the sweet spot for on-device Apple Silicon deployment. |
| `group_size` | `32` | Each group of 32 weights shares one quantization scale/exponent. Smaller groups (32 vs 64 or 128) give finer-grained calibration and better accuracy, at a small storage overhead. **For QAT, prefer group size 32** — the model has a chance to learn within these tighter bins. |
| `mode` | `"mxfp4"` | MX (Microscaling) FP4 is the native Apple Silicon low-precision format. It uses a shared exponent per group rather than a uniform integer grid, preserving more of the dynamic range of the original weights. Prefer `mxfp4` over `int4` for M-series chips. |

In [ ]:
model_name = "allenai/Olmo-3-7B-Instruct" # The base model to fine-tune.
new_model_name = "Olmo-3-7B-Instruct-mxfp4-QAT" # The name for the fine-tuned, QAT model with LoRA applied.
adapter_path = f"./{new_model_name}" # The path to save the LoRA adapter. This is a small file that contains the fine-tuned weights and can be merged with the base model for inference.
user_name = "mlx-community" # Hugging Face username, needed if you want to push the model to the Hugging Face Hub. You can create an account for free at https://huggingface.co/join

dataset_name = "mlx-community/Dolci-Instruct-SFT-No-Tools-100K" # The dataset to fine-tune on.

max_seq_length = 4096 # Increase or decrease based on your dataset and RAM capacity. Longer sequences require more VRAM but can capture more context.

lora_config = { # LoRA adapter configuration
    "rank": 12,  # Low-rank bottleneck size (Larger rank = smarter, but slower). Suggested 8, 16, 32, 64, 128
    "dropout": 0.0,
    "scale": 10.0, # Multiplier for how hard the LoRA update hits the base weights
    "use_dora": False, # Use DoRA, which is a more efficient version of LoRA that uses a single matrix instead of two.
    "num_layers": 10 # Use -1 for all layers
}
quantized_config={
    "bits": 4, # Use 4 bit quantization. Suggested 4, 6, 8
    "group_size": 32, # Quantize in groups of 32 weights. Smaller group size means better performance but slower inference. Suggested 32, 64, 128
    "mode": "mxfp4", # Quantization mode. "mxfp4" is a good balance between performance and accuracy.
}

## Step 3 — Load the Model and Create the LoRA Adapter

`from_pretrained` does three things in one call:

1. **Downloads and loads the base model** from the Hugging Face Hub (or local cache) using MLX.
2. **Quantizes it immediately** to `mxfp4 4-bit` using the `quantized_load` config, so the weights land in GPU/unified memory already in their compressed form. This is what makes the 7B model fit comfortably under 16 GB of RAM.
3. **Injects LoRA adapter layers** according to `lora_config` and writes an empty `adapter.safetensors` file to `adapter_path`. The adapter file grows as training checkpoints are saved.

`print_trainable_parameters` then confirms how many of the total parameters are actually trainable. Expect to see only ~0.1–0.5% of total params marked as trainable — the rest of the 7B weights are frozen in their quantized form. This is what keeps memory and training time manageable.

In [ ]:
# Load the model, tokenizer, and create/save the adapter path and file.
model, tokenizer, adapter_file = from_pretrained(
    model=model_name,
    new_adapter_path=adapter_path,
    lora_config=lora_config,
    quantized_load=quantized_config,
)
print_trainable_parameters(model)

## Step 4 — Format the Dataset with the Chat Template

Raw datasets store conversations as structured `messages` lists (role/content pairs). The model's tokenizer includes a **chat template** — a Jinja2 template that serialises those lists into the exact token sequence the model was pre-trained on (system turn, user turn, assistant turn, special tokens).

Key argument: `enable_thinking=False`  
OLMo-3 Instruct supports an optional "thinking" mode that inserts chain-of-thought reasoning tokens. We disable it here because the training dataset does not contain thinking-format examples — enabling it on mismatched data would add noise rather than signal.

In [ ]:
# The formatting function applies the chat template to the messages in the dataset, which is necessary for the model to understand the input correctly.
def format(sample):
    sample["text"] = tokenizer.apply_chat_template(
        conversation=sample["messages"],
        add_generation_prompt=False,
        tokenize=False,
        enable_thinking=False
    )
    return sample

## Step 5 — Load and Prepare the Dataset

[`mlx-community/Dolci-Instruct-SFT-No-Tools-100K`](https://huggingface.co/datasets/mlx-community/Dolci-Instruct-SFT-No-Tools-100K) is a curated 100K instruction-following dataset, stripped of tool-call examples so the model learns clean conversational instruction following. Crucially, **this is the same dataset used to fine-tune the base OLMo-3 7B Instruct model**, which is exactly what we want for QAT — the model replays its original training distribution while simultaneously adapting its weights to quantization constraints.

**`.take(100)` / `.take(10)`** — Limits to a fast smoke-test subset so the notebook completes in minutes on any M-series chip. Remove `.take()` to train on the full dataset for a production-quality result.

**`TextDataset`** tokenizes each formatted string and packs the token IDs up to `max_seq_length`. It masks the prompt tokens from the loss so the model only learns to predict the assistant's reply, not the user's question.

**`CacheDataset`** (used at training time) pre-tokenizes and caches all sequences in unified memory, so the training loop never stalls waiting for the CPU tokenizer. Always wrap your datasets in `CacheDataset` when training — it removes tokenisation from the critical path.

In [ ]:
# Load the dataset, apply the formatting function, and create the TextDataset objects for training and validation. We take a subset of the dataset for quick training, but you can remove the .take() calls to use the full dataset.
train_dataset = load_dataset(dataset_name, split="train").take(100).map(format)
valid_dataset = load_dataset(dataset_name, split="valid").take(10).map(format)

# The TextDataset class is a simple wrapper that tokenizes the text and prepares it for training.
train_set = TextDataset(train_dataset, tokenizer, text_key="text")
valid_set = TextDataset(valid_dataset, tokenizer, text_key="text")

In [ ]:
# Just to check everything is working, let's print the first formatted text from the training dataset.
print(train_dataset[0]["text"])

## Step 6 — Baseline Inference (Before Training)

Run the model against a maths problem **before any QAT fine-tuning**. This gives you a reference point — you can qualitatively compare reasoning quality, answer correctness, and output style against the post-training result in Step 8.

The prompt combines two sub-problems (geometric series sum + binomial probability) to test multi-step reasoning, which tends to degrade most visibly under naive 4-bit PTQ.

In [ ]:
prompt = "Find the sum of the first 6 terms of a geometric sequence with first term a=4 and common ratio r=3, then calculate the probability of getting exactly 2 heads in 5 coin tosses."

text = tokenizer.apply_chat_template(
    conversation=[{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    tokenize=False,
    enable_thinking=False
)

In [ ]:
_ = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=text,
    verbose=True,
    max_tokens=512
)

## Step 7 — Quantization-Aware Training (QAT + LoRA)

### Optimizer — `AdamW` at `lr=2e-5`
AdamW (Adam with decoupled weight decay) is the standard choice for LoRA fine-tuning. A low learning rate of `2e-5` prevents the small LoRA matrices from over-writing the generalisation already baked into the base model. If loss plateaus early, try `5e-5`; if it oscillates, go lower to `1e-5`.

### SFT training arguments

| Argument | Value | Reasoning |
|---|---|---|
| `batch_size` | `1` | Fits within 16 GB unified memory. Effective batch size is amplified by gradient accumulation below. |
| `iters` | auto | `calculate_iters` sets this to `⌈len(train_set) / batch_size⌉ × epochs` — exactly 1 full pass for quick testing. |
| `gradient_accumulation_steps` | `8` | Simulates a batch size of 8 without holding 8 samples in memory simultaneously. Larger effective batch = more stable QAT gradient signal. |
| `val_batches` | `1` | Only 1 validation batch since we have a tiny validation split. Increase for a full evaluation. |
| `steps_per_report` | `10` | Prints training loss every 10 steps — enough resolution to spot divergence early. |
| `steps_per_eval` | `20` | Runs validation every 20 steps. |
| `steps_per_save` | `10` | Saves the adapter checkpoint every 10 steps, giving fine-grained recovery points. |
| `max_seq_length` | `4096` | Caps token sequences at 4096 to bound memory use. Sequences longer than this are truncated. |
| `grad_checkpoint` | `True` | Gradient checkpointing trades compute for memory: activations are recomputed during the backward pass instead of cached. Enables training a 7B model on <16 GB at the cost of ~20% slower backward pass. **Keep enabled unless you have 32+ GB RAM.** |

### QAT-specific arguments

| Argument | Value | Reasoning |
|---|---|---|
| `qat_enable` | `True` | Activates the QAT simulation wrapper around the model. |
| `qat_bits` | `4` | Must match the target quantization depth in your `quantized_config`. |
| `qat_group_size` | `32` | Must match `quantized_config["group_size"]`. The STE operates at the same granularity as inference quantization, so mismatching these would simulate the wrong noise distribution. |
| `qat_mode` | `"mxfp4"` | Must match `quantized_config["mode"]`. Tells the STE to use the MX FP4 rounding rules. |
| `qat_start_step` | `1` | QAT begins on the very first step. Starting immediately forces the LoRA adapter to learn under quantization from the start, rather than first learning in float and then switching. This is recommended when using the original training data. |
| `qat_interval` | `1` | Re-quantizes every single step. Setting this higher (e.g. `5`) speeds up training slightly but the model sees fewer quantization-noise gradients, reducing QAT benefit. Keep at `1` for best quality. |

> **Tip — Scaling up:** If you have 32+ GB RAM, remove `.take()` on the datasets, increase `iters` to 3–5 epochs, set `batch_size=2` and reduce `gradient_accumulation_steps` to 4. With the full 100K dataset you will see a measurable perplexity improvement over PTQ.

In [ ]:
opt = optim.AdamW(learning_rate=2e-5) # Use a low learning rate for LoRA fine-tuning, since we don't want to change the base model weights too much. Adjust if necessary.

# Training arguments. Adjust these based on your dataset size, GPU capacity, and how long you want to train.
args = SFTTrainingArgs(
    batch_size=1, # Use batch size of 1 to save RAM, increase if you have more GPU memory
    iters=calculate_iters(train_set, batch_size=1, epochs=1), # Only train for 1 epoch since the dataset is small, increase if you want to train longer
    gradient_accumulation_steps=8, # Accumulate gradients over 8 steps to simulate a larger batch size and save RAM. Adjust based on your GPU capacity.
    val_batches=1, # Only use 1 batch for validation to speed it up, since the dataset is small. Remove or increase for better evaluation.
    steps_per_report=10, # Log training progress every 10 steps
    steps_per_eval=20, # Evaluate every 20 steps
    steps_per_save=10, # Save the model every 10 steps
    max_seq_length=max_seq_length,
    adapter_file=adapter_file,
    grad_checkpoint=True, # Use gradient checkpointing to save RAM at the cost of slightly slower training
    # seq_step_size=512, # This makes it efficient, increasing it makes th emodel learn better but uses more RAM. Suggested 512, 1024, 2048

    # This enables Quantization Aware Training (QAT), which simulates the effects of quantization during training to help the model adapt to it and achieve better performance when quantized. Adjust the parameters based on your needs and GPU capacity.
    qat_enable=True,
    qat_bits=quantized_config["bits"],
    qat_group_size=quantized_config["group_size"],
    qat_mode=quantized_config["mode"],
    qat_start_step=1, # Start QAT from the beginning of training. You can increase this if you want to start QAT after some initial training steps.
    qat_interval=1 # Apply QAT every step. You can increase this to apply QAT every few steps to save some training time, at the cost of slightly less accurate quantization simulation.
)

# Start training
train_sft(
    model=model,
    args=args,
    optimizer=opt,
    train_dataset=CacheDataset(train_set),
    val_dataset=CacheDataset(valid_set),
    # training_callback=WandBCallback(
    #     project_name=f"{new_model_name}-finetuning",
    #     log_dir=adapter_path,
    #     wrapped_callback=TrainingCallback(),
    #     config=None
    # )
    training_callback=TrainingCallback(), # You can use the basic TrainingCallback to log training progress to the console instead of Weights & Biases. Just comment out the WandBCallback and uncomment this line if you prefer that.
)

## Step 8 — Post-Training Inference

Run the same prompt again with the QAT-tuned model. Compare the output against the baseline from Step 6.

What to look for:
- **Correctness** — Does it arrive at the right numeric answers?
- **Reasoning structure** — Does it show step-by-step working, or does it jump straight to a number?
- **Coherence** — Are sentences well-formed with no repetition or gibberish (a common sign of aggressive quantization damage)?

Even with only 100 training samples the QAT model typically shows noticeably more stable reasoning than the same model quantized via PTQ alone.

In [ ]:
_ = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=text,
    verbose=True,
    max_tokens=1024
)

## Step 9 — Save the Merged Model

`save_pretrained_merged` fuses the LoRA adapter deltas back into the base weight matrices and writes the result to `adapter_path`.

Two important flags for a QAT model:

**`de_quantize=False`** — Keep the weights in their quantized `mxfp4` form. This is the whole point of QAT: the saved model is small, fast to load, and ready for on-device inference without a separate quantization step. Setting `de_quantize=True` would reconstruct float16 weights, discarding the compression and the QAT benefit.

**`remove_adapters=False`** — Retains the LoRA adapter weights alongside the merged weights. This means the saved model can still be further fine-tuned with LoRA in future runs. Set to `True` only if you want the smallest possible deployment artefact and have no plans to continue fine-tuning.

In [ ]:
save_pretrained_merged(
    model=model,
    tokenizer=tokenizer,
    save_path=adapter_path,
    de_quantize=False, # Since its a QAT model, we save it in quantized form. Set to True if you want to de-quantize it back to normal weights, but it will lose the benefits of QAT and be larger in size.
    remove_adapters=False # Keep the adapter weights in the saved model. Set to True if you want to remove the adapter weights and only save the merged model, but it will lose the ability to further fine-tune with LoRA and be larger in size.
)

## Step 10 — Push to the Hugging Face Hub

Upload the final model to your Hub repository so it can be shared, versioned, and loaded directly by other `mlx-lm` users.

Replace `"HF_KEY"` with your actual Hugging Face write token (generate one at https://huggingface.co/settings/tokens). Never hard-code real tokens in shared notebooks — prefer reading from an environment variable or a secrets manager.

**`remove_adapters=True`** here (vs `False` in the save step) — when publishing, you typically want to ship a clean merged model without the raw adapter files, keeping the upload smaller and the repository self-contained for end users who just want to do inference.

In [ ]:
push_to_hub(
  model_path=f"./{new_model_name}",
  hf_repo=f"{user_name}/{new_model_name}",
  api_key="HF_KEY",
  private=False,
  commit_message="Add fine-tuned LoRA adapter",
  remove_adapters=True
)

---

## Summary

You have just produced a **4-bit mxfp4 QAT fine-tuned OLMo-3 7B Instruct** model that:

- Fits and runs efficiently on Apple Silicon with <16 GB unified memory
- Was trained with Quantization-Aware Training so its weights are adapted to the MX FP4 quantization grid — not just snapped to it after the fact
- Uses the same instruction-following dataset as the original base model, maximising quality recovery under quantization
- Has LoRA adapter weights merged in, ready for direct inference with `mlx_lm.generate` or any compatible `mlx-lm` tooling

**Next steps:**
- Remove `.take()` and train on the full 100K dataset for a production-quality model
- Increase `rank` to 16 or 32 and `num_layers` to `-1` for more expressive fine-tuning
- Enable the `WandBCallback` to track loss curves and compare runs
- Try `qat_start_step=50` to let the LoRA adapter warm up in float before QAT noise is applied — useful if you observe instability at the start of training